# 🖥️ Cloud Desktop — Google Colab

[![GitHub](https://img.shields.io/badge/GitHub-M5Develop%2FCloud--Desktop-181717?logo=github)](https://github.com/M5Develop/Cloud-Desktop-Colab-kaggle)
[![License: GPL v3](https://img.shields.io/badge/License-GPLv3-blue.svg)](https://github.com/M5Develop/Cloud-Desktop-Colab-kaggle/blob/main/LICENSE)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/M5Develop/Cloud-Desktop-Colab-kaggle/blob/main/kaggle_desktop.ipynb)

Run a full XFCE4 desktop on Google Colab and access it from any browser — no VPN, no credit card, no local storage needed.

**Stack:** `Xvfb` + `XFCE4` + `x11vnc` + `noVNC` + `localhost.run` + `Brave Browser`

**Notes:**
- ✅ Colab (Xubuntu) already has XFCE4, x11vnc, and websockify pre-installed
- ✅ Brave installs without `snap`
- ⏱️ Free tier session lasts ~12 hours

**Use cases:**
- 📤 Upload large model files (7GB+) to CivitAI using Colab's bandwidth — not yours
- 🖥️ Run any GUI application on a free cloud environment
- 🌐 Browse the web through a remote Linux desktop

> ⚠️ Run cells **in order**, one by one.

## 📦 Cell 1 — Install Dependencies
> Colab already has most packages pre-installed — this just ensures everything is ready.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq xfce4 x11vnc xvfb dbus-x11 novnc websockify openssh-client 2>/dev/null
print('✅ All dependencies ready!')

## 🌐 Cell 2 — Install Brave Browser
> Installs via official `.deb` — no `snap` required.

In [ ]:
import subprocess
subprocess.run('curl -fsS https://dl.brave.com/install.sh | sh', shell=True)
print('✅ Brave installed!')

## 🖥️ Cell 3 — Start Virtual Display + XFCE4

In [ ]:
import subprocess, os, time

os.environ['DISPLAY'] = ':99'

subprocess.Popen('Xvfb :99 -screen 0 1280x800x24 -ac', shell=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
print('✅ Xvfb running!')

subprocess.Popen('DISPLAY=:99 startxfce4', shell=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)
print('✅ XFCE4 running!')

## 📡 Cell 4 — Start VNC Server

In [ ]:
import subprocess, time

VNC_PASSWORD = '12345678'  # Change if you want

subprocess.run(f'x11vnc -storepasswd {VNC_PASSWORD} /root/.vncpasswd', shell=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

subprocess.Popen(
    'x11vnc -display :99 -rfbauth /root/.vncpasswd -rfbport 5900 -forever -loop -shared -noxdamage',
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print(f'✅ VNC server running (password: {VNC_PASSWORD})')

## 🖱️ Cell 5 — Start noVNC

In [ ]:
import subprocess, time
subprocess.Popen('websockify --web=/usr/share/novnc/ 6080 localhost:5900', shell=True,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print('✅ noVNC running on port 6080!')

## 🔗 Cell 6 — Create Public Tunnel via localhost.run
> ✅ Free | ✅ No account | ✅ No credit card | Uses SSH only

In [ ]:
import subprocess, time

subprocess.Popen(
    'ssh -o StrictHostKeyChecking=no -o ServerAliveInterval=60 '
    '-R 80:localhost:6080 nokey@localhost.run '
    '> /root/tunnel.log 2>&1',
    shell=True
)

print('⏳ Starting tunnel...')
time.sleep(15)

with open('/root/tunnel.log', 'r') as f:
    content = f.read()

found = False
for line in content.split('\n'):
    if 'localhost.run' in line and 'https://' in line:
        for word in line.split():
            if (word.startswith('https://') and 'localhost.run' in word
                    and not any(x in word for x in ['admin.', '/docs'])):
                print('=' * 55)
                print('🎉 Your remote desktop is ready!')
                print(f'🌐 Open: {word.strip()}/vnc.html')
                print(f'🔒 Password: {VNC_PASSWORD}')
                print('=' * 55)
                found = True
                break

if not found:
    print('⚠️ URL not ready — run Cell 7 to check the log')

## 🔍 Cell 7 — Show Full Tunnel Log (if URL didn't appear)

In [ ]:
with open('/root/tunnel.log', 'r') as f:
    print(f.read())

## 🚀 Cell 8 — Launch Brave Browser on the Desktop

In [ ]:
import subprocess, os
subprocess.Popen('DISPLAY=:99 brave-browser --no-sandbox', shell=True,
    env={**os.environ, 'DISPLAY': ':99'},
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
print('✅ Brave launched on the desktop!')

## ⬇️ Cell 9 (Optional) — Download Model from HuggingFace
> Server-to-server download — **zero local bandwidth used!** 😎

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

REPO_ID  = 'your-username/your-model'   # ← Change this
FILENAME = 'your-model.safetensors'     # ← Change this

print(f'⏳ Downloading {FILENAME} from HuggingFace...')
print('(Server-to-server — no local bandwidth used 😎)')

path = hf_hub_download(repo_id=REPO_ID, filename=FILENAME, local_dir='/root/')

os.makedirs('/root/Desktop', exist_ok=True)
shutil.copy(path, f'/root/Desktop/{FILENAME}')

print(f'✅ Downloaded: {path}')
print('✅ Copied to Desktop — ready to upload!')

## 🛑 Cell 10 — Stop Everything

In [ ]:
import subprocess
subprocess.run('pkill x11vnc; pkill Xvfb; pkill xfce4-session; pkill websockify; pkill ssh', shell=True)
print('🛑 All processes stopped!')